# FlyWire Error Analysis: Synapse Count Measurement Experiment

**One Kaggle Run = One Dataset + One Error Model + One Analysis Profile**

---
### How to run on Kaggle
1. Attach both datasets to this notebook:
   - `flywire-codebase` (uploaded from `flywire_codebase.zip`)
   - `flywire-all-datasets` (uploaded from `flywire_all_datasets.zip`)
2. In **Cell 3**, set `DATASET_NAME` to whichever connectome you want to run.
3. Click **Run All**.

### Kaggle Dataset Paths (fixed, no changes needed)
- Codebase : `/kaggle/input/datasets/jeet7771/flywire-codebase`
- Data      : `/kaggle/input/datasets/jeet7771/flywire-all-datasets`

In [ ]:
# Cell 1: Environment Setup & sys.path
# This MUST run before any framework imports.
# ============================================================
import os
import sys
from pathlib import Path

IS_KAGGLE = os.path.exists('/kaggle/input')

# Exact Kaggle dataset mount paths for user jeet7771
KAGGLE_CODEBASE_PATH = Path('/kaggle/input/datasets/jeet7771/flywire-codebase')
KAGGLE_DATA_PATH     = Path('/kaggle/input/datasets/jeet7771/flywire-all-datasets')

if IS_KAGGLE:
    # --- Verify and add codebase to sys.path ---
    if not KAGGLE_CODEBASE_PATH.exists():
        raise FileNotFoundError(
            f'Codebase dataset not found at {KAGGLE_CODEBASE_PATH}\n'
            'Attach the "flywire-codebase" dataset to this notebook.'
        )
    sys.path.insert(0, str(KAGGLE_CODEBASE_PATH))
    print(f'[OK] Codebase path  : {KAGGLE_CODEBASE_PATH}')

    # --- Verify data dataset ---
    if not KAGGLE_DATA_PATH.exists():
        raise FileNotFoundError(
            f'Data dataset not found at {KAGGLE_DATA_PATH}\n'
            'Attach the "flywire-all-datasets" dataset to this notebook.'
        )
    print(f'[OK] Data path      : {KAGGLE_DATA_PATH}')
    print(f'[OK] Datasets found : {[d.name for d in KAGGLE_DATA_PATH.iterdir() if d.is_dir()]}')

else:
    # Local: codebase is the current working directory
    REPO_ROOT = Path(os.getcwd())
    if str(REPO_ROOT) not in sys.path:
        sys.path.insert(0, str(REPO_ROOT))
    print(f'[OK] Running locally. Repo root: {REPO_ROOT}')

print(f'Environment: {"KAGGLE" if IS_KAGGLE else "LOCAL"}')

In [ ]:
# Cell 2: Framework Imports
import warnings
import pandas as pd

warnings.filterwarnings('ignore')

from core.experiment_runner import ExperimentRunner, ExperimentConfig
from modules.error_models import registry as error_registry
from modules.graph_analyses.analysis_registry import registry as analysis_registry
from modules.statistical_evaluation import StatisticalEvaluator
from core.export_manager import ExportManager

print('All framework imports successful.')

In [ ]:
# ============================================================
# Cell 3: RUNTIME CONFIGURATION  <-- ONLY CELL YOU NEED TO EDIT
# ============================================================

# Which connectome to run.
# Options: "BANC" | "FAFB" | "MANC" | "MAOL" | "MCNS" | "TEST"
DATASET_NAME = "BANC"

# [LOCAL ONLY] Path to your raw dataset folder. Ignored on Kaggle.
LOCAL_DATASET_ROOT = "research_data/raw"

# NOTE: For this model, error_rate means RELATIVE MEASUREMENT UNCERTAINTY,
# not fraction of edges modified.  E.g. error_rate=0.05 means
# sigma = 0.05 * original_weight for every edge.

EXPERIMENT = {
    "metadata": {
        "experiment_name": f"SynapseCount_{DATASET_NAME}",
        "author": "FlyWire Researcher",
        "description": "Impact of synapse count measurement uncertainty on weighted graph analyses.",
    },
    "error": {
        "name": "synapse_count_measurement",
        # Error rates here = relative measurement uncertainty %
        "rates": [
            0.000,    # 0%   — baseline (no noise)
            0.005,    # 0.5%
            0.010,    # 1%
            0.020,    # 2%
            0.030,    # 3%
            0.050,    # 5%
            0.075,    # 7.5%
            0.100,    # 10%
            0.150,    # 15%
            0.200,    # 20%
        ],
        "random_seeds": [1, 2, 3, 4, 5],
    },
    # No biology section needed for this model.
    # The measurement model operates on every edge independently.
    "analysis": [
        "basic_structure",       # total_synapses, weight_mean, weight_std → SHOULD change
        "degree_distribution",   # topological — control (should NOT change)
        "pagerank",              # weighted — SHOULD change
        "assortativity",         # topological — control (should NOT change)
        # "centrality",           # HEAVY — keep disabled for runtime profiling
        "connected_components",  # topological — control (should NOT change)
        "reciprocity",           # topological — control (should NOT change)
    ],
    "export": {
        "create_zip": True,
        "save_statistics": True,
    },
}

OUTPUT_ROOT = Path("results") / DATASET_NAME / EXPERIMENT["error"]["name"]

print(f'Dataset Name    : {DATASET_NAME}')
print(f'Error Model     : {EXPERIMENT["error"]["name"]}')
print(f'Error Rates      : {EXPERIMENT["error"]["rates"]}   (measurement uncertainty)')
print(f'Trials per Rate : {len(EXPERIMENT["error"]["random_seeds"])}')
print(f'Output Root     : {OUTPUT_ROOT}')

In [ ]:
# Cell 4: Resolve Dataset Root
if IS_KAGGLE:
    DATASET_ROOT = str(KAGGLE_DATA_PATH)
else:
    DATASET_ROOT = '0-demodata' if DATASET_NAME.upper() == 'TEST' else LOCAL_DATASET_ROOT

print(f'DATASET_ROOT = {DATASET_ROOT}')

In [ ]:
# Cell 5: Verify Dataset Structure
from core.dataset_registry import DatasetRegistry, DatasetRegistryError

CONFIGS_ROOT = str(KAGGLE_CODEBASE_PATH / 'configs') if IS_KAGGLE else 'configs'

try:
    reg = DatasetRegistry(configs_root=CONFIGS_ROOT, dataset_root=DATASET_ROOT)
    resolved_dir = reg.resolve_dataset_dir(DATASET_NAME, DATASET_ROOT)
    print(f'[OK] Dataset "{DATASET_NAME}" verified.')
    print(f'     Resolved: {resolved_dir}')
except DatasetRegistryError as e:
    raise FileNotFoundError(
        f'Cannot resolve dataset "{DATASET_NAME}" in "{DATASET_ROOT}".\n'
        f'Expected a subfolder named {DATASET_NAME}_<version>/ or {DATASET_NAME}/.\n'
        f'Error: {e}'
    ) from e

In [ ]:
# Cell 6: Verify Registries
err_model = EXPERIMENT['error']['name']
print(f'Registered Error Models : {error_registry.list_names()}')
print(f'Registered Analyses     : {analysis_registry.list_names()}')

assert err_model in error_registry.list_names(), \
    f'Error model "{err_model}" not registered.'
missing = [a for a in EXPERIMENT['analysis'] if a not in analysis_registry.list_names()]
assert not missing, f'Analyses not registered: {missing}'

print('[OK] All required components registered. Ready to run.')

In [ ]:
# Cell 7: Run Experiments
runner = ExperimentRunner(analysis_registry, error_registry)
results_per_rate = {}

for err_rate in EXPERIMENT['error']['rates']:
    rate_str = f"{err_rate * 100:g}".replace('.', '_') + "_percent"
    results_per_rate[err_rate] = []

    for trial, seed in enumerate(EXPERIMENT['error']['random_seeds'], 1):
        print(f'\n{"="*50}')
        print(f'  Dataset    : {DATASET_NAME}')
        print(f'  Error Rate : {err_rate * 100:g}%  (uncertainty)')
        print(f'  Trial      : {trial} / {len(EXPERIMENT["error"]["random_seeds"])}')
        print(f'  Seed       : {seed}')
        print(f'{"="*50}')

        trial_out = OUTPUT_ROOT / rate_str / f'trial_{trial:03d}'

        config = ExperimentConfig(
            dataset_name=DATASET_NAME,
            dataset_root=str(DATASET_ROOT),
            configs_root=CONFIGS_ROOT,
            error_model_name=err_model,
            error_model_config={
                'error_rate': err_rate,
            },
            analysis_names=EXPERIMENT['analysis'],
            preprocessing_config={'features': {'degree': True, 'synapse_counts': True}},
            seed=seed,
            output_root=str(trial_out) if EXPERIMENT['export']['save_statistics'] else None,
            create_zip=EXPERIMENT['export']['create_zip'],
            extra={'metadata': EXPERIMENT['metadata']},
        )

        res = runner.run(config)
        results_per_rate[err_rate].append(res)

        if res.succeeded:
            meta = res.error_result.perturbation_metadata if res.error_result else {}
            pct_changed = meta.get('pct_edges_changed', 0)
            print(f'  --> Success! ({res.runtime_seconds:.2f}s) edges_changed={pct_changed:.1f}%')
        else:
            print(f'  --> FAILED!  Errors: {res.errors}')

print('\nAll trials complete.')

In [ ]:
# Cell 8: Statistical Evaluation
evaluator = StatisticalEvaluator()
aggregated_stats_by_rate = {}

baseline_runs = [r for r in results_per_rate.get(0.00, []) if r.succeeded]
if not baseline_runs:
    raise RuntimeError('No successful baseline (0%) runs. Cannot evaluate.')

for err_rate, run_results in results_per_rate.items():
    successful = [r for r in run_results if r.succeeded]
    if successful:
        eval_result = evaluator.evaluate(baseline_runs, successful)
        aggregated_stats_by_rate[err_rate] = eval_result
        print(f'Evaluated {err_rate*100:g}%  -> {len(successful)} successful trials')
    else:
        print(f'Skipped   {err_rate*100:g}%  -> 0 successful trials')

print('\nStatistical evaluation complete.')

In [ ]:
# Cell 9: Export Presentation Layer
# Plots -> results/<DATASET>/synapse_count_measurement/presentation/plots/
ExportManager().export_presentation(
    results_by_rate=aggregated_stats_by_rate,
    output_root=OUTPUT_ROOT,
    metadata=EXPERIMENT['metadata'],
)
print(f'Presentation exported to : {OUTPUT_ROOT / "presentation"}')
print(f'Plots saved to           : {OUTPUT_ROOT / "presentation" / "plots"}')

In [ ]:
# Cell 10: Quick Summary
print('=' * 50)
print(f'  {DATASET_NAME} - {err_model}')
print('=' * 50)
for err_rate in sorted(aggregated_stats_by_rate.keys()):
    ev = aggregated_stats_by_rate[err_rate]
    print(f'  Uncertainty {err_rate*100:g}%')
    for a_name, metrics in ev.metrics.items():
        print(f'    {a_name}: {len(metrics)} metrics')
        for m_name, m_dict in list(metrics.items())[:3]:
            print(f'      {m_name}: mean={m_dict.mean:.4f} d={m_dict.effect_size:.4f}')
total_trials = sum(len(v) for v in results_per_rate.values())
print(f'  Trials: {total_trials} | Rates: {len(aggregated_stats_by_rate)}')
print(f'  Output: {OUTPUT_ROOT}')
print('=' * 50)